Imports

In [1]:
import sys
import os
package_path = os.path.abspath("../..")  
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-05-26 18:26:19.175065: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-26 18:26:19.184436: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-26 18:26:19.184460: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

In [3]:
scm.helloworld()

hello world!


Make the dask cluster & client in accordance with resource avail and model size

In [19]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=4:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)
cluster.scale(jobs=1)

In [9]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [36]:
#let's make an ortho object
test=scm.ortho()
test.criss_cross(client=client,dat=dat)

In [ ]:
test.extract_params(client)

In [39]:
test.by_cre_parameters.result()

(mean parameter             brain       blood
 C(cre_id)[everybody]  107.685347  103.586014
 C(cre_id)[neurogene]   96.254215   15.500235
 C(cre_id)[nobody]       1.022865    1.945718
 C(cre_id)[redgene]     30.724592  105.632993
 C(cre_id)[somebody]     9.995359   12.492922,
 zero inflation fraction     brain     blood
 C(rep_id)[1]             0.793373  0.796269
 C(rep_id)[2]             0.505266  0.508310
 C(rep_id)[3]             0.899705  0.907928,
 brain    1.3041227
 blood     1.199084
 dtype: object)

In [40]:
nb,zi,theta=model_to_parameters(unif.result())

In [51]:
working=unif.result().uniq_predictor['unified'][0]
working=working.sort_values(working.columns.to_list(),ascending=False)

In [54]:
working

,C(cre_id)[everybody],C(cre_id)[neurogene],C(cre_id)[nobody],C(cre_id)[redgene],C(cre_id)[somebody],C(cell_type)[T.brain],C(cre_id)[T.neurogene]:C(cell_type)[T.brain],C(cre_id)[T.nobody]:C(cell_type)[T.brain],C(cre_id)[T.redgene]:C(cell_type)[T.brain],C(cre_id)[T.somebody]:C(cell_type)[T.brain]
904,1,0,0,0,0,1,0,0,0,0
3266,1,0,0,0,0,0,0,0,0,0
1798,0,1,0,0,0,1,1,0,0,0
4262,0,1,0,0,0,0,0,0,0,0
0,0,0,1,0,0,1,0,1,0,0
2263,0,0,1,0,0,0,0,0,0,0
1355,0,0,0,1,0,1,0,0,1,0
3767,0,0,0,1,0,0,0,0,0,0
440,0,0,0,0,1,1,0,0,0,1
2762,0,0,0,0,1,0,0,0,0,0


In [ ]:
working.columns[]

In [18]:
cluster.close()